In [ ]:
import pandas as pd


In [8]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_15108\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


In [9]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Imputing missing values ##

In [ ]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [15]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [13]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [16]:
df.drop(['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [17]:
df.sample(10)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
488265,681,7,2014-04-20,0,0,0,0,0,0,a,c,620.0,1,2014,4,0,15.0,16,3.75,0
660983,574,6,2013-11-16,8997,723,1,0,0,0,d,a,4400.0,0,2013,11,0,10.0,46,0.00,0
43497,13,1,2015-06-22,2603,220,1,0,0,0,d,a,310.0,1,2015,6,0,29.0,26,67.25,0
418325,986,7,2014-06-22,0,0,0,0,0,0,a,a,620.0,1,2014,6,0,0.0,25,1.75,0
86417,563,5,2015-05-15,5088,645,1,0,0,0,a,a,700.0,1,2015,5,0,2.0,20,14.50,0
413448,569,4,2014-06-26,4053,629,1,0,0,0,a,a,1340.0,0,2014,6,0,93.0,26,0.00,0
523811,547,3,2014-03-19,5664,426,1,1,0,0,d,c,8990.0,1,2014,3,0,52.0,12,42.25,1
204054,10,4,2015-01-29,6203,603,1,1,0,0,a,a,3160.0,0,2015,1,0,64.0,5,0.00,0
536565,1036,6,2014-03-08,4813,466,1,0,0,0,d,c,9560.0,1,2014,3,0,14.0,10,5.50,0
596430,691,1,2014-01-13,6066,501,1,0,0,0,d,c,3030.0,1,2014,1,0,12.0,3,51.50,1
